# 01 - Data Transformation on Databricks

This demo walks through the main ways to transform data on Databricks and when to pick each one.

**Prerequisite:** run `00_Setup_Mock_Data` first and use the same **catalog** / **schema** widget values here.

**Demo assets**
* `00_Setup_Mock_Data`: schema, volumes and mock data (`orders/`, `customers_cdc/`, `events/`, `products/`)
* `01_Data_Transformation_Overview` (this notebook): concepts + batch PySpark / SQL
* `02_Structured_Streaming_Auto_Loader`: hand-written Structured Streaming
* `pipeline/`: Lakeflow Spark Declarative Pipeline (bronze/silver/gold, expectations, AUTO CDC), deployed with `databricks.yml`
* `03_Lakeflow_Connect`: managed ingestion connectors

## The Lakeflow family
| Component | What it does |
| --- | --- |
| **Lakeflow Connect** | Managed, no-code ingestion from SaaS apps (Salesforce, Workday, ServiceNow...) and databases (SQL Server, PostgreSQL, MySQL...) plus standard connectors (Auto Loader, Kafka, files) |
| **Lakeflow Spark Declarative Pipelines (SDP)** | Declarative ETL in SQL/Python: streaming tables, materialized views, expectations, AUTO CDC; the engine handles orchestration, checkpoints, retries, incremental refresh |
| **Lakeflow Jobs** | Orchestration of notebooks, pipelines, SQL, dbt, etc. with triggers (schedule, file arrival, table update) |

## Choosing a transformation approach

| Approach | Best for | Limitations / watch-outs |
| --- | --- | --- |
| **PySpark DataFrame API (batch)** | Complex logic, reusable functions, ML feature prep, unit-testable code | You own orchestration, idempotency, incremental logic (watermarks, MERGE), and retries |
| **Databricks SQL (notebooks / SQL warehouse)** | Analysts & analytics engineers, ELT on curated tables, MERGE/REPLACE WHERE, standalone MVs/STs | Less suited to complex procedural logic; Python UDFs are slower than native functions |
| **Structured Streaming (hand-written)** | Full control of streaming: custom `foreachBatch` sinks, multi-sink writes, low-latency, stateful ops (`transformWithState`) | You manage checkpoints, schema evolution, restarts, DQ, dependency ordering; more code to operate |
| **Lakeflow SDP** | Medallion pipelines, incremental ingest with Auto Loader/Kafka, CDC/SCD1/SCD2 via AUTO CDC, built-in data quality (expectations), lineage & observability | Declarative model: no side effects in dataset functions (no `.collect()`, `.write`); tables are pipeline-owned (use sinks for external targets); changing a dataset type (ST↔MV) needs a new name; some features are Python-only (sinks, foreachBatch sinks) |
| **Lakeflow Connect (managed connectors)** | Getting data *in* from SaaS/DBs with incremental sync, schema evolution and API rate-limit handling, minimal code | Connector-specific limits (supported objects, API quotas, DB CDC prerequisites, gateway compute for DBs); transformations happen downstream (SDP/SQL) |
| **dbt on Databricks** | Teams standardized on dbt for SQL modeling & testing | Batch-oriented; streaming is better served by SDP |

**Rules of thumb**
* Incremental files/streams + medallion + DQ -> **SDP**
* Need full control or custom sinks/side effects -> **Structured Streaming** (or SDP + `foreach_batch_sink`)
* SaaS / operational DB source -> **Lakeflow Connect** first, then transform with SDP
* Ad-hoc / exploratory / ML -> **PySpark or SQL in notebooks**, then productionize

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "de_demo", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
spark.sql("USE CATALOG IDENTIFIER(:c)", args={"c": CATALOG})
spark.sql("USE SCHEMA IDENTIFIER(:s)", args={"s": SCHEMA})

# Temp views over the raw files with read_files(), so the SQL cells below stay catalog-agnostic
spark.sql(f"CREATE OR REPLACE TEMP VIEW raw_orders AS SELECT * FROM read_files('{LANDING}/orders', format => 'json')")
spark.sql(f"CREATE OR REPLACE TEMP VIEW raw_products AS SELECT * FROM read_files('{LANDING}/products', format => 'csv', header => true)")
spark.sql(f"CREATE OR REPLACE TEMP VIEW raw_customers_cdc AS SELECT * FROM read_files('{LANDING}/customers_cdc', format => 'json')")
display(dbutils.fs.ls(LANDING))

## 1. Batch transformation with PySpark
Read raw JSON, clean it, enrich with a dimension, aggregate and write a Delta table. Data quality, idempotency and incremental processing are all our responsibility here.

In [0]:
from pyspark.sql import functions as F

orders_raw = spark.read.format("json").load(f"{LANDING}/orders")
products = (spark.read.format("csv").option("header", True).option("inferSchema", True)
            .load(f"{LANDING}/products"))

# Manual data quality: keep only valid rows
is_valid = F.col("customer_id").isNotNull() & (F.col("amount") >= 0)
orders_clean = (orders_raw.filter(is_valid)
                .withColumn("order_ts", F.to_timestamp("order_ts"))
                .withColumn("order_date", F.to_date("order_ts")))
raw_n, clean_n = orders_raw.count(), orders_clean.count()
print(f"raw={raw_n}  clean={clean_n}  rejected={raw_n - clean_n}")

daily_sales = (orders_clean.join(products, "product_id")
               .groupBy("order_date", "category")
               .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))

# Full overwrite = simple & idempotent, but recomputes everything each run
daily_sales.write.mode("overwrite").saveAsTable("pyspark_daily_sales")
display(spark.table("pyspark_daily_sales").orderBy("order_date", "category"))

## 2. Same logic in SQL
`read_files()` (wrapped in the `raw_*` temp views above) + CTAS. SQL is often the most accessible option for analytics engineers; the same SQL can later become an SDP materialized view.

In [0]:
%sql
CREATE OR REPLACE TABLE sql_daily_sales AS
WITH orders AS (
  SELECT *, to_date(order_ts) AS order_date
  FROM raw_orders
  WHERE customer_id IS NOT NULL AND amount >= 0
)
SELECT o.order_date, p.category, count(*) AS orders, round(sum(o.amount), 2) AS revenue
FROM orders o JOIN raw_products p USING (product_id)
GROUP BY ALL;

SELECT * FROM sql_daily_sales ORDER BY order_date, category;

In [0]:
%sql
-- Both approaches should produce identical results (expect 0 rows)
(SELECT * FROM pyspark_daily_sales EXCEPT SELECT * FROM sql_daily_sales)
UNION ALL
(SELECT * FROM sql_daily_sales EXCEPT SELECT * FROM pyspark_daily_sales)

## 3. Incremental upserts by hand (MERGE)
The CDC feed in `customers_cdc/` has INSERT/UPDATE/DELETE events. By hand you must dedupe by key, order by sequence, handle deletes and out-of-order data yourself. Compare with the few-line `AUTO CDC` flow in `pipeline/transformations/silver/sdp_customers.py`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_manual (customer_id BIGINT, name STRING, email STRING, country STRING, tier STRING, updated_at TIMESTAMP);

MERGE INTO customers_manual t
USING (
  -- keep only the latest event per key
  SELECT customer_id, name, email, country, tier, operation, CAST(updated_at AS TIMESTAMP) AS updated_at
  FROM raw_customers_cdc
  QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1
) s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET name = s.name, email = s.email, country = s.country, tier = s.tier, updated_at = s.updated_at
WHEN NOT MATCHED AND s.operation <> 'DELETE' THEN INSERT (customer_id, name, email, country, tier, updated_at)
  VALUES (s.customer_id, s.name, s.email, s.country, s.tier, s.updated_at);

SELECT tier, count(*) AS customers FROM customers_manual GROUP BY tier ORDER BY tier;